# 02 — Build the Silver layer: `drug/event` (month-by-month, scratch on D:)

Flattens bronze to the clean atomic **(case, drug, reaction)** grain and applies **#4 / #5 / #6**.

**Safe on a small laptop:** 
- **All Spark scratch goes to D:, never C:.** Spill dir is `/opt/spark-tmp` (mounted to
  `D:/capstone/data/spark-tmp` via `SPARK_LOCAL_DIRS`), and the Parquet cache is
  `/home/jovyan/dq_cache` (mounted to `D:/capstone/data/dq_cache`). Orphaned spill on C: is what
  filled the C: drive before — it can't anymore.
- **Month-by-month.** Instead of one giant `dropDuplicates` + `partitionBy` shuffle over the whole
  dataset (which OOM'd the driver), we process one month at a time and write each partition. Each
  batch is tiny, so memory stays low and a crash can't strand tens of GB of spill.
- **JSON -> Parquet once.** Bronze JSON is cached to Parquet a single time; every silver build reads
  the Parquet, never re-parses the nested JSON.

**This version adds (data-quality hardening):**
- **Latest version only** — keep the highest `safetyreportversion` per `safetyreportid` (explicit guard).
- **Validation #6 extended** — quarantine null/blank **reaction** *and* null/blank **drug name**, not
  just bad `drugcharacterization`. No null reaction ever enters Silver.
- **Metrics persisted** — a `silver_metrics` Parquet (one row per month) with input/atomic/silver/
  duplicates/resolved/quarantined counts.
- **Explicit clean-rebuild** — `CLEAN_REBUILD` wipes prior Silver/Quarantine/metrics *with a printed
  message* before rebuilding (never silent), so no stale partitions linger.
- **Honest #5 label** — the resolution rate is "resolved via the report's own generic/substance name",
  NOT full pharmaceutical identity. Event-level ids (`rxcui`, `product_ndc`, `package_ndc`,
  `brand_name`) are carried through for the later **dbt** NDC join (`int_drug_resolution`, `dim_drug`).

In [1]:
import os
# Driver memory the reliable way (before the JVM starts). Batches are small now, so 4g is plenty.
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 4g pyspark-shell'

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName('openfda_silver')
         .config('spark.driver.memory', '4g')
         .config('spark.sql.shuffle.partitions', '64')          # small monthly batches
         .config('spark.sql.parquet.enableVectorizedReader', 'false')  # safe on deeply-nested cols
         .config('spark.local.dir', '/opt/spark-tmp')           # spill to D: (also set via SPARK_LOCAL_DIRS)
         .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')  # overwrite one month at a time
         .getOrCreate())
print('spark.local.dir        :', spark.conf.get('spark.local.dir'))
print('SPARK_LOCAL_DIRS (env) :', os.environ.get('SPARK_LOCAL_DIRS'))
spark

spark.local.dir        : /opt/spark-tmp
SPARK_LOCAL_DIRS (env) : /opt/spark-tmp


## 0. Config — paths (all on D:), months, code maps, clean-rebuild switch

In [2]:
import datetime, shutil

BRONZE_DIR     = '/home/jovyan/data/bronze/drug_event'   # read-only (D:)
CACHE_DIR      = '/home/jovyan/dq_cache/drug_event'       # Parquet cache (D:)
SILVER_OUT     = '/home/jovyan/silver/drug_event'         # Silver output (D:)
QUAR_OUT       = '/home/jovyan/quarantine/drug_event'     # Quarantine (D:)
SILVER_METRICS = '/home/jovyan/silver/_silver_metrics'    # per-month run metrics (D:)

MONTHS = [(y, m) for y in (2023, 2024) for m in range(1, 13)]  # 2023-01 .. 2024-12
RUN_ID = 'silver_' + datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')

# EXPLICIT clean rebuild: wipe prior Silver/Quarantine/metrics before rebuilding (prints what it does).
# Only touches DERIVED outputs on D: — bronze is read-only and never affected.
CLEAN_REBUILD = True

print('RUN_ID:', RUN_ID, '| months:', len(MONTHS), '| CLEAN_REBUILD:', CLEAN_REBUILD)

SEX              = {'0': 'Unknown', '1': 'Male', '2': 'Female'}
CHARACTERIZATION = {'1': 'SUSPECT', '2': 'CONCOMITANT', '3': 'INTERACTING'}
VALID_CHAR       = ['1', '2', '3']
OUTCOME          = {'1': 'Recovered/resolved', '2': 'Recovering/resolving',
                    '3': 'Not recovered/not resolved', '4': 'Recovered with sequelae',
                    '5': 'Fatal', '6': 'Unknown'}
QUALIFICATION    = {'1': 'Physician', '2': 'Pharmacist', '3': 'Other health professional',
                    '4': 'Lawyer', '5': 'Consumer or non-health professional'}

def decode(col, mapping, default=None):
    items = list(mapping.items())
    expr = F.when(col == items[0][0], F.lit(items[0][1]))
    for code_, label in items[1:]:
        expr = expr.when(col == code_, F.lit(label))
    return expr.otherwise(F.lit(default).cast('string'))

def flag(name):
    return F.when(F.col(name) == '1', True).otherwise(False)

# Slim each drug element BEFORE exploding. Keep the event-level identifiers we will need later for
# the dbt NDC join (rxcui, product_ndc, package_ndc, brand_name) — carried through, not matched here.
SLIM_DRUGS = ("transform(patient.drug, d -> struct("
              "d.medicinalproduct as medicinalproduct, "
              "d.drugcharacterization as drugcharacterization, "
              "element_at(d.openfda.generic_name, 1) as generic_name, "
              "element_at(d.openfda.substance_name, 1) as substance_name, "
              "element_at(d.openfda.rxcui, 1) as rxcui, "
              "element_at(d.openfda.product_ndc, 1) as product_ndc, "
              "element_at(d.openfda.package_ndc, 1) as package_ndc, "
              "element_at(d.openfda.brand_name, 1) as brand_name))")

RUN_ID: silver_20260805T151033Z | months: 24 | CLEAN_REBUILD: True


## 1. JSON -> Parquet cache (once), month by month

Reads each month's bronze JSON and writes it to a Parquet partition on D:. Runs only if the cache
is missing. Each month is small, so memory stays low and spill (if any) goes to D:.

In [3]:
def month_prefix(y, m):
    return '%d%02d' % (y, m)

built = os.path.isdir(CACHE_DIR) and any(d.startswith('receive_year=') for d in os.listdir(CACHE_DIR))
if built:
    print('cache already present ->', CACHE_DIR)
else:
    print('building Parquet cache (one-time), month by month...')
    for (y, m) in MONTHS:
        src = BRONZE_DIR + '/receivedate=' + month_prefix(y, m) + '*/*.json'
        try:
            mdf = spark.read.json(src)
        except Exception:
            print('  skip', y, m, '(no files)'); continue
        (mdf.withColumn('receive_year', F.lit(y)).withColumn('receive_month', F.lit(m))
            .write.mode('overwrite').partitionBy('receive_year', 'receive_month').parquet(CACHE_DIR))
        print('  cached', month_prefix(y, m))
    print('cache build done.')

cache already present -> /home/jovyan/dq_cache/drug_event


## 2. Build Silver, month by month (#4 dedup · #5 normalise · #6 validate/quarantine)

`build_month` returns the pre-dedup Silver rows and the quarantine rows for one month. The loop
dedups, writes each month's partition (dynamic overwrite → idempotent), and records metrics.

In [4]:
GRAIN = ['safety_report_id', 'resolved_drug', 'drug_characterization_code', 'reaction_pt']

def build_month(mdf):
    # --- latest version only: keep highest safetyreportversion per safetyreportid (explicit guard;
    #     openFDA already returns latest-only, verified 0 dup in exploration — this enforces it) ---
    w = Window.partitionBy('safetyreportid').orderBy(F.col('safetyreportversion').cast('int').desc_nulls_last())
    mdf = mdf.withColumn('_rn', F.row_number().over(w)).filter(F.col('_rn') == 1).drop('_rn')

    reports = mdf.select(
        F.col('safetyreportid').alias('safety_report_id'),
        F.col('safetyreportversion').cast('int').alias('report_version'),
        F.to_date('receivedate', 'yyyyMMdd').alias('receive_date'),
        F.to_date('receiptdate', 'yyyyMMdd').alias('receipt_date'),
        F.col('serious'),
        F.col('seriousnessdeath'), F.col('seriousnesshospitalization'),
        F.col('seriousnesslifethreatening'), F.col('seriousnessdisabling'),
        F.col('seriousnesscongenitalanomali'), F.col('seriousnessother'),
        F.col('primarysource.qualification').alias('reporter_qualification_code'),
        F.col('occurcountry').alias('occur_country'),
        F.col('primarysourcecountry').alias('primary_source_country'),
        F.col('patient.patientsex').alias('patientsex'),
        F.col('patient.patientonsetage').alias('patientonsetage'),
        F.col('patient.patientonsetageunit').alias('patientonsetageunit'),
        F.expr(SLIM_DRUGS).alias('drugs'),
        F.col('patient.reaction').alias('reactions'),
    )
    drug_rows = reports.select('*', F.posexplode('drugs').alias('drug_idx', 'drug')).drop('drugs')

    # --- #6 drug-grain validation: bad drugcharacterization OR blank drug name -> quarantine ---
    char  = F.col('drug.drugcharacterization')
    dname = F.col('drug.medicinalproduct')
    drug_reject = (F.when(char.isNull(), F.lit('drugcharacterization_null'))
                    .when(~char.isin(*VALID_CHAR), F.lit('drugcharacterization_out_of_range'))
                    .when(dname.isNull() | (F.trim(dname) == ''), F.lit('drug_name_blank'))
                    .otherwise(F.lit(None).cast('string')))
    drug_rows = drug_rows.withColumn('_dreject', drug_reject)

    quar_drug = (drug_rows.filter(F.col('_dreject').isNotNull())
        .select('safety_report_id', 'report_version',
                dname.alias('medicinalproduct_raw'),
                char.alias('drug_characterization_code'),
                F.lit(None).cast('string').alias('reaction_pt'),
                F.col('_dreject').alias('_reject_reason'),
                F.year('receive_date').alias('receive_year'),
                F.month('receive_date').alias('receive_month')))

    valid = drug_rows.filter(F.col('_dreject').isNull()).drop('_dreject')
    # scatter within-report skew (a mega-report's many drug rows) before the reaction explode
    valid = valid.repartition(64, F.col('safety_report_id'), F.col('drug_idx'))
    atomic = valid.select('*', F.explode('reactions').alias('reaction')).drop('reactions')

    # --- #6 reaction-grain validation: blank reaction term -> quarantine (never enters Silver) ---
    rpt = F.col('reaction.reactionmeddrapt')
    react_blank = rpt.isNull() | (F.trim(rpt) == '')
    quar_react = (atomic.filter(react_blank)
        .select('safety_report_id', 'report_version',
                F.col('drug.medicinalproduct').alias('medicinalproduct_raw'),
                char.alias('drug_characterization_code'),
                rpt.alias('reaction_pt'),
                F.lit('reaction_pt_null').alias('_reject_reason'),
                F.year('receive_date').alias('receive_year'),
                F.month('receive_date').alias('receive_month')))

    quarantine = quar_drug.unionByName(quar_react)
    good = atomic.filter(~(rpt.isNull() | (F.trim(rpt) == '')))

    # --- #5 normalise (honest: resolves via the report's OWN generic/substance name) ---
    raw_name  = F.col('drug.medicinalproduct')
    generic   = F.col('drug.generic_name')
    substance = F.col('drug.substance_name')
    # Tier-2 cleanup: uppercase, strip obvious dosage tokens (e.g. 100MG), drop punctuation, collapse spaces.
    name_up     = F.upper(raw_name)
    name_nodose = F.regexp_replace(name_up, '[0-9]+ ?(MG|MCG|UG|G|GM|ML|L|IU|MEQ|MMOL|UNITS|UNIT|%)', ' ')
    name_clean  = F.trim(F.regexp_replace(F.regexp_replace(name_nodose, '[^A-Z0-9 ]', ' '), ' +', ' '))
    resolved = (F.when(generic.isNotNull()   & (F.trim(generic) != ''),   F.upper(F.trim(generic)))
                 .when(substance.isNotNull() & (F.trim(substance) != ''), F.upper(F.trim(substance)))
                 .otherwise(name_clean))
    tier = (F.when(generic.isNotNull()   & (F.trim(generic) != ''),   F.lit('generic_name'))
             .when(substance.isNotNull() & (F.trim(substance) != ''), F.lit('substance_name'))
             .otherwise(F.lit('unresolved_raw')))
    age  = F.col('patientonsetage').cast('double')
    unit = F.col('patientonsetageunit')
    age_years = (F.when(unit == '800', age * 10)
                  .when((unit == '801') | unit.isNull(), age)
                  .when(unit == '802', age / 12).when(unit == '803', age / 52)
                  .when(unit == '804', age / 365).when(unit == '805', age / 8760)
                  .otherwise(F.lit(None)))
    age_band = (F.when(age_years.isNull() | (age_years < 0) | (age_years > 120), F.lit('Unknown'))
                 .when(age_years < 18, F.lit('0-17')).when(age_years < 45, F.lit('18-44'))
                 .when(age_years < 65, F.lit('45-64')).when(age_years < 75, F.lit('65-74'))
                 .otherwise(F.lit('75+')))

    silver = good.select(
        F.col('safety_report_id'), F.col('report_version'),
        F.col('receive_date'), F.col('receipt_date'),
        raw_name.alias('medicinalproduct_raw'), name_clean.alias('drug_name_clean'),
        resolved.alias('resolved_drug'), tier.alias('drug_resolution_tier'),
        (tier != 'unresolved_raw').alias('drug_resolved'),
        F.col('drug.rxcui').alias('rxcui'),
        F.col('drug.product_ndc').alias('product_ndc'),
        F.col('drug.package_ndc').alias('package_ndc'),
        F.col('drug.brand_name').alias('brand_name'),
        char.alias('drug_characterization_code'), decode(char, CHARACTERIZATION).alias('drug_characterization'),
        F.col('reaction.reactionmeddrapt').alias('reaction_pt'),
        F.col('reaction.reactionmeddraversionpt').alias('reaction_meddra_version'),
        F.col('reaction.reactionoutcome').alias('reaction_outcome_code'),
        decode(F.col('reaction.reactionoutcome'), OUTCOME).alias('reaction_outcome'),
        F.when(F.col('serious') == '1', True).when(F.col('serious') == '2', False)
         .otherwise(F.lit(None).cast('boolean')).alias('is_serious'),
        flag('seriousnessdeath').alias('outcome_death'),
        flag('seriousnesshospitalization').alias('outcome_hospitalisation'),
        flag('seriousnesslifethreatening').alias('outcome_life_threatening'),
        flag('seriousnessdisabling').alias('outcome_disability'),
        flag('seriousnesscongenitalanomali').alias('outcome_congenital_anomaly'),
        flag('seriousnessother').alias('outcome_other'),
        F.col('reporter_qualification_code'),
        decode(F.col('reporter_qualification_code'), QUALIFICATION).alias('reporter_type'),
        F.coalesce(F.col('occur_country'), F.lit('Unknown')).alias('occur_country'),
        F.coalesce(F.col('primary_source_country'), F.lit('Unknown')).alias('primary_source_country'),
        F.coalesce(decode(F.col('patientsex'), SEX), F.lit('Unknown')).alias('patient_sex'),
        age_band.alias('patient_age_band'),
        F.year('receive_date').alias('receive_year'), F.month('receive_date').alias('receive_month'),
        F.lit(RUN_ID).alias('_run_id'), F.current_timestamp().alias('_loaded_at'),
        F.sha2(F.concat_ws('||', F.col('safety_report_id'), resolved, char,
                           F.col('reaction.reactionmeddrapt')), 256).alias('report_drug_reaction_key'),
    )
    return silver, quarantine

# EXPLICIT clean-rebuild (logged, never silent) — only derived outputs on D:
if CLEAN_REBUILD:
    for p in (SILVER_OUT, QUAR_OUT, SILVER_METRICS):
        if os.path.isdir(p):
            print('clean-rebuild: removing existing output ->', p)
            shutil.rmtree(p)

month_stats = []
for (y, m) in MONTHS:
    part = CACHE_DIR + '/receive_year=' + str(y) + '/receive_month=' + str(m)
    if not os.path.isdir(part):
        print('  no cache partition for', y, m, '- skipping'); continue
    mdf = spark.read.parquet(part)
    silver_m, quar_m = build_month(mdf)
    n_atomic = silver_m.count()
    (silver_m.dropDuplicates(GRAIN).write.mode('overwrite')
        .partitionBy('receive_year', 'receive_month').parquet(SILVER_OUT))
    n_quar = quar_m.count()
    (quar_m.write.mode('overwrite')
        .partitionBy('receive_year', 'receive_month').parquet(QUAR_OUT))
    month_stats.append((y, m, int(n_atomic), int(n_quar)))
    print('  silver written:', y, m, '| atomic', format(n_atomic, ','), '| quarantined', n_quar)

# --- persist per-month metrics ---
silver_all = spark.read.parquet(SILVER_OUT)
per_month = (silver_all.groupBy('receive_year', 'receive_month')
             .agg(F.count(F.lit(1)).alias('silver_rows'),
                  F.sum(F.col('drug_resolved').cast('int')).alias('resolved_rows')))
atomic_df = spark.createDataFrame(month_stats, ['receive_year', 'receive_month', 'atomic_rows', 'quarantined_rows'])
metrics = (atomic_df.join(per_month, ['receive_year', 'receive_month'], 'left')
           .withColumn('duplicates_removed', F.col('atomic_rows') - F.col('silver_rows'))
           .withColumn('run_id', F.lit(RUN_ID))
           .withColumn('run_timestamp', F.current_timestamp())
           .orderBy('receive_year', 'receive_month'))
metrics.write.mode('overwrite').parquet(SILVER_METRICS)
print('all months done. metrics ->', SILVER_METRICS)

  silver written: 2023 1 | atomic 2,760,603 | quarantined 6
  silver written: 2023 2 | atomic 2,669,968 | quarantined 1
  silver written: 2023 3 | atomic 2,927,431 | quarantined 25000
  silver written: 2023 4 | atomic 3,307,732 | quarantined 0
  silver written: 2023 5 | atomic 4,325,002 | quarantined 0
  silver written: 2023 6 | atomic 2,725,979 | quarantined 1
  silver written: 2023 7 | atomic 2,564,571 | quarantined 0
  silver written: 2023 8 | atomic 3,893,681 | quarantined 2
  silver written: 2023 9 | atomic 3,660,418 | quarantined 0
  silver written: 2023 10 | atomic 3,740,172 | quarantined 61249
  silver written: 2023 11 | atomic 3,194,395 | quarantined 0
  silver written: 2023 12 | atomic 3,276,171 | quarantined 0
  silver written: 2024 1 | atomic 3,682,443 | quarantined 0
  silver written: 2024 2 | atomic 4,095,307 | quarantined 0
  silver written: 2024 3 | atomic 4,686,131 | quarantined 0
  silver written: 2024 4 | atomic 4,285,524 | quarantined 1
  silver written: 2024 5 | at

## 3. Results — data-quality summary (#4 / #5 / #6)

In [5]:
metrics = spark.read.parquet(SILVER_METRICS)
silver_all = spark.read.parquet(SILVER_OUT)
q_out = spark.read.parquet(QUAR_OUT)

t = metrics.agg(F.sum('atomic_rows').alias('a'), F.sum('silver_rows').alias('s'),
                F.sum('quarantined_rows').alias('q'), F.sum('resolved_rows').alias('r')).first()
a, s, q, r = t['a'], t['s'], t['q'], t['r']
print('atomic rows (pre-dedup) :', format(a, ','))
print('silver rows (deduped)   :', format(s, ','))
print('duplicates removed (#4) :', format(a - s, ','), '(' + format(100.0 * (a - s) / a, '.2f') + '%)')
print()
print('#5 resolved via the report\'s OWN generic/substance name :', format(100.0 * r / s, '.2f') + '%')
print('   (this is NOT full NDC/rxcui resolution — that is the later dbt step)')
silver_all.groupBy('drug_resolution_tier').count().orderBy(F.desc('count')).show(truncate=False)
print('#6 quarantined entries  :', q)
q_out.groupBy('_reject_reason').count().orderBy(F.desc('count')).show(truncate=False)

atomic rows (pre-dedup) : 93,366,638
silver rows (deduped)   : 45,030,932
duplicates removed (#4) : 48,335,706 (51.77%)

#5 resolved via the report's OWN generic/substance name : 78.46%
   (this is NOT full NDC/rxcui resolution — that is the later dbt step)
+--------------------+--------+
|drug_resolution_tier|count   |
+--------------------+--------+
|generic_name        |35330227|
|unresolved_raw      |9700705 |
+--------------------+--------+

#6 quarantined entries  : 431760
+---------------------------------+------+
|_reject_reason                   |count |
+---------------------------------+------+
|reaction_pt_null                 |431741|
|drugcharacterization_out_of_range|18    |
|drugcharacterization_null        |1     |
+---------------------------------+------+



## 4. Verify — guardrails (24 months, no null reactions, unique grain, reconciled counts)

In [6]:
nparts = silver_all.select('receive_year', 'receive_month').distinct().count()
print('month partitions present  :', nparts, '(expected 24)')

nnull = silver_all.filter(F.col('reaction_pt').isNull() | (F.trim(F.col('reaction_pt')) == '')).count()
print('null/blank reaction in silver:', nnull, '(expected 0)')

rows = silver_all.count()
d = silver_all.select('report_drug_reaction_key').distinct().count()
print('grain key unique          :', d == rows, '(' + format(rows, ',') + ' / ' + format(d, ',') + ')')

print('null resolved_drug        :', silver_all.filter(F.col('resolved_drug').isNull()).count())

# reconcile: silver rows from metrics == actual silver rows
m_silver = metrics.agg(F.sum('silver_rows').alias('s')).first()['s']
print('reconcile metrics vs disk :', m_silver == rows, '(' + format(m_silver, ',') + ' / ' + format(rows, ',') + ')')

silver_all.select('safety_report_id', 'resolved_drug', 'brand_name', 'rxcui',
                  'drug_characterization', 'reaction_pt', 'patient_sex', 'patient_age_band').show(8, truncate=False)

month partitions present  : 24 (expected 24)
null/blank reaction in silver: 0 (expected 0)
grain key unique          : True (45,030,932 / 45,030,932)
null resolved_drug        : 0
reconcile metrics vs disk : True (45,030,932 / 45,030,932)
+----------------+---------------------------+----------------------------+-------+---------------------+------------------------+-----------+----------------+
|safety_report_id|resolved_drug              |brand_name                  |rxcui  |drug_characterization|reaction_pt             |patient_sex|patient_age_band|
+----------------+---------------------------+----------------------------+-------+---------------------+------------------------+-----------+----------------+
|21803134        |DUPILUMAB                  |DUPIXENT                    |1876401|SUSPECT              |Product dispensing issue|Male       |0-17            |
|21803136        |TRISODIUM CITRATE DIHYDRATE|ANTICOAGULANT SODIUM CITRATE|NULL   |SUSPECT              |Hyperhidrosis   

In [7]:
print('=== per-month metrics ===')
metrics.select('receive_year', 'receive_month', 'atomic_rows', 'silver_rows',
               'duplicates_removed', 'resolved_rows', 'quarantined_rows').show(24)
silver_all.printSchema()

=== per-month metrics ===
+------------+-------------+-----------+-----------+------------------+-------------+----------------+
|receive_year|receive_month|atomic_rows|silver_rows|duplicates_removed|resolved_rows|quarantined_rows|
+------------+-------------+-----------+-----------+------------------+-------------+----------------+
|        2023|            1|    2760603|    1549263|           1211340|      1233397|               6|
|        2023|            2|    2669968|    1509602|           1160366|      1201763|               1|
|        2023|            3|    2927431|    1588696|           1338735|      1273494|           25000|
|        2023|            4|    3307732|    1813797|           1493935|      1443153|               0|
|        2023|            5|    4325002|    2121226|           2203776|      1670914|               0|
|        2023|            6|    2725979|    1546443|           1179536|      1214016|               1|
|        2023|            7|    2564571|    141

In [8]:
q = spark.read.parquet('/home/jovyan/quarantine/drug_event').filter("_reject_reason='reaction_pt_null'")
print('distinct reports affected:', q.select('safety_report_id').distinct().count())
q.groupBy('safety_report_id').count().orderBy(F.desc('count')).show(10, truncate=False)

distinct reports affected: 3
+----------------+------+
|safety_report_id|count |
+----------------+------+
|23840947        |345492|
|23014826        |61249 |
|22122822        |25000 |
+----------------+------+



In [9]:
spark.read.parquet('/home/jovyan/dq_cache/drug_event') \
  .filter("safetyreportid in ('23840947','23014826','22122822')") \
  .selectExpr('safetyreportid','reporttype','size(patient.drug) as n_drugs','size(patient.reaction) as n_reactions').show()

+--------------+----------+-------+-----------+
|safetyreportid|reporttype|n_drugs|n_reactions|
+--------------+----------+-------+-----------+
|      23014826|         1|   2663|         23|
|      22122822|         1|   1000|         25|
|      23840947|         1|   4113|         84|
+--------------+----------+-------+-----------+



## Summary

| Issue | Action | Result |
|---|---|---|
| **#4 Dedup** | latest version per case, then drop duplicates at (case, drug, reaction) | _see metrics_ |
| **#5 Normalisation** | Tier-1 report `generic_name`/`substance_name`, Tier-2 cleaned raw (dosage/punct stripped) | _resolution rate (report-level)_ |
| **#6 Validation** | quarantine bad `drugcharacterization`, blank drug name, **blank reaction**; demographics -> Unknown; age banded | _quarantine count_ |

**Honest scope:** `resolved_drug` is resolved via the **report's own** `generic_name`/`substance_name`
(or a cleaned raw fallback) — **not** a full pharmaceutical identity. Event-level ids (`rxcui`,
`product_ndc`, `package_ndc`, `brand_name`) are carried through for the later **dbt** NDC join
(`int_drug_resolution`, `dim_drug`).

**Outputs (D:):** `silver/drug_event/` (Parquet, partitioned by year/month), `quarantine/drug_event/`,
`silver/_silver_metrics/`. All Spark scratch stays on D:; run is month-by-month; clean-rebuild is explicit.